# Contact analysis

As its accronym suggests, MuJoCo has been developped to handle contacts with a modern approach. Indeed, the engine uses new formulation of the physics contact (see the original [publication](https://ieeexplore.ieee.org/document/6386109))

Let's see how to analyse contacts in MuJoCo.

## Render contact forces

Contact always involve two geometries defined in the model.
Contact generation in MuJoCo is an elaborate process. First one has to detect collision between to geometries and then estimate the different contact parameters (normal, force...) according to the chosen contact model. See details in the [Documentation](https://mujoco.readthedocs.io/en/stable/XMLreference.html#contact).

Let's start with a very simple model: a box falling on the ground.

**Questions:**
- Complete the model: define a box of side 0.1m
- Add a `<freejoint/>` to give the body all degrees of freedoms.

In [ ]:
import mujoco
import mujoco.viewer
import time
import numpy as np

import matplotlib.pyplot as plt

free_body_MJCF = """
<mujoco>
	<asset>
		<texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3"
		rgb2=".2 .3 .4" width="300" height="300" mark="edge" markrgb=".2 .3 .4"/>
		<material name="grid" texture="grid" texrepeat="2 2" texuniform="true"
		reflectance=".2"/>
	</asset>

	<worldbody>
		<light pos="0 0 1" mode="trackcom"/>
		<geom name="ground" type="plane" pos="0 0 -.5" size="2 2 .1" material="grid" solimp=".99 .99 .01" solref=".001 1"/>
		<body name="box" pos="0 0 0">
			/// TODO
			<freejoint/>
			<geom name="red_box" type="box" size=".1 .1 .1" euler="45 40 30" rgba="1 0 0 1"/>
			/// TODO
		</body>
	</worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(free_body_MJCF)
data = mujoco.MjData(model)

Let's visualize the contact points and the contact forces. To do so in the viewer, we set the flags `mjVIS_CONTACTPOINT`, `mjVIS_CONTACTFORCE` to `True`.

Press *space* to make the box fall again.

**Questions:**
- Randomize the orientation of the box everytime you press the spacebar.

In [ ]:
MAX_SIM_TIME = 30

def sim_viewer(model,
               data,
               max_sim_time=MAX_SIM_TIME):
	
	free_fall = True
	
	def key_callback(keycode):
		if chr(keycode) == ' ':
			nonlocal free_fall
			free_fall = not free_fall
		
	with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
		
		# Visualize contact
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True
		
		# tweak scales of contact visualization elements
		model.vis.scale.contactwidth = 0.1
		model.vis.scale.contactheight = 0.03
		model.vis.scale.forcewidth = 0.05
		model.vis.map.force = 0.3
		
		# Close the viewer automatically after 30 wall-seconds.
		start = time.time()
		while viewer.is_running() and time.time() - start < max_sim_time:
			step_start = time.time()
			
			if free_fall:
				free_fall = not free_fall
				data.qpos[:3] = np.zeros(3)
				# TODO
				q = np.random.rand(4) # random orientation
				q /= np.linalg.norm(q)
				data.qpos[3:7] = q
			
			# mj_step can be replaced with code that also evaluates
			# a policy and applies a control signal before stepping the physics.
			mujoco.mj_step(model, data)

			# Pick up changes to the physics state, apply perturbations, update options from GUI.
			viewer.sync()

			# Rudimentary time keeping, will drift relative to wall clock.
			time_until_next_step = model.opt.timestep - (time.time() - step_start)
			if time_until_next_step > 0:
				time.sleep(time_until_next_step)
				
sim_viewer(model, data)

### Analysis of contact forces

Data related to the contacts are stored in `data.contact`. One can iterate over the contact pairs to extract the data needed. For instance:

- `geom1`: The name of the first geom in the pair.
- `geom2`: The name of the second geom in the pair.
- `friction`: The friction coefficients (float [5]).

See the [documentation](https://mujoco.readthedocs.io/en/stable/XMLreference.html#contact) for more information.

For instance, let's plot the number of contact, the normal force of the contact as well as the penetration depth (how deep the object goes through the ground).

**Questions:**
- Get the contact force using [`mj_contactForce`](https://mujoco.readthedocs.io/en/stable/APIreference/APIfunctions.html#mj-contactforce) and fill the corresponding data array.
- Get the distance to contact and fill the corresponding data array.


In [ ]:
def plot_contacts(model,
				  data,
				  max_sim_time=1.4):
	n_steps = int(max_sim_time / model.opt.timestep)

	# allocate
	sim_time = np.zeros(n_steps)
	ncon = np.zeros(n_steps)
	force = np.zeros((n_steps,3))    # Sum of all contact forces
	penetration = np.zeros(n_steps)  # Penetration distance of the contact
	forcetorque = np.zeros(6)

	with mujoco.viewer.launch_passive(model, data) as viewer:
	
		# Visualize contact
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True
		
		# tweak scales of contact visualization elements
		model.vis.scale.contactwidth = 0.1
		model.vis.scale.contactheight = 0.03
		model.vis.scale.forcewidth = 0.05
		model.vis.map.force = 0.3
		
		# Close the viewer automatically after 30 wall-seconds.
		i = 0
		start = time.time()
		while viewer.is_running() and i < n_steps:
			step_start = time.time()
			
			# mj_step can be replaced with code that also evaluates
			# a policy and applies a control signal before stepping the physics.
			mujoco.mj_step(model, data)

			# Pick up changes to the physics state, apply perturbations, update options from GUI.
			viewer.sync()

			# Rudimentary time keeping, will drift relative to wall clock.
			time_until_next_step = model.opt.timestep - (time.time() - step_start)
			if time_until_next_step > 0:
				time.sleep(time_until_next_step)
				
			# iterate over active contacts, save force and distance
			for j, c in enumerate(data.contact):
				mujoco.mj_contactForce(model, data, j, forcetorque)
				force[i] += forcetorque[0:3] # TODO: linear force at contact
				pen_dist = c.dist            # TODO: penetration distance
				penetration[i] = min(penetration[i], pen_dist)
				
			# Fill data arrays
			sim_time[i] = data.time
			ncon[i] = data.ncon
			i += 1
			
	# plot
	_, ax = plt.subplots(2, 2, sharex=True, figsize=(10, 10))

	lines = ax[0,0].plot(sim_time, force)
	ax[0,0].set_title('contact force')
	ax[0,0].set_ylabel('Newton')
	ax[0,0].legend(iter(lines), ('normal z', 'friction x', 'friction y'));

	ax[0,1].plot(sim_time, ncon)
	ax[0,1].set_title('number of contacts')
	ax[0,1].set_yticks(range(6))

	ax[1,0].plot(sim_time, force[:,0])
	ax[1,0].set_yscale('log')
	ax[1,0].set_title('normal (z) force - log scale')
	ax[1,0].set_ylabel('Newton')
	z_gravity = -model.opt.gravity[2]
	mg = model.body("box").mass[0] * z_gravity
	mg_line = ax[1,0].plot(sim_time, np.ones(n_steps)*mg, label='m*g', linewidth=1)
	ax[1,0].legend()

	ax[1,1].plot(sim_time, 1000*penetration)
	ax[1,1].set_title('penetration depth')
	ax[1,1].set_ylabel('millimeter')
	ax[1,1].set_xlabel('second')

	plt.tight_layout()
	plt.show()
	
model = mujoco.MjModel.from_xml_string(free_body_MJCF)
data = mujoco.MjData(model)

# Random orientation
data.qpos[:3] = np.zeros(3)
q = np.random.rand(4)
q /= np.linalg.norm(q)
data.qpos[3:7] = q

plot_contacts(model, data)

The `friction` parameters can be changed in the model.
On the following example, the box has an intial linear velocity.
When the friction increase (e.g. `FRICTION = 2`) the cube will start rolling while for a lower friction (`FRICTION = 0.1`) the box slides on the floor.

In [ ]:
FRICTION = 2.

free_body_MJCF_friction = f"""
<mujoco>
  <asset>
	<texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3"
	rgb2=".2 .3 .4" width="300" height="300" mark="edge" markrgb=".2 .3 .4"/>
	<material name="grid" texture="grid" texrepeat="2 2" texuniform="true"
	reflectance=".2"/>
  </asset>

  <worldbody>
	<light pos="0 0 1" mode="trackcom"/>
	<geom name="ground" type="plane" pos="0 0 -0.12" size="2 2 .1" material="grid" solimp=".99 .99 .01" solref=".001 1"/>
	<body name="box" pos="0 0 0">
	  <freejoint/>
	  <geom name="red_box" friction="{FRICTION}" type="box" size=".1 .1 .1" euler="0 0 0" rgba="1 0 0 1" solimp=".99 .99 .01"  solref=".001 1"/>
	  <camera name="fixed" pos="0 -.6 .3" xyaxes="1 0 0 0 1 2"/>
	  <camera name="track" pos="0 -.6 .3" xyaxes="1 0 0 0 1 2" mode="track"/>
	</body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(free_body_MJCF_friction)
data = mujoco.MjData(model)

# Initial linear velocity
data.qpos[2] = 0.001
data.qvel[1] = 3.

plot_contacts(model, data, 2)